In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # moves from src/models to Classify_Number folder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

path = Path.cwd().parents[1]

In [7]:
path

PosixPath('/home/nguyenhaianh/Desktop/Classify_Number')

In [8]:
import numpy as np
import torch

from src.models.ensemble_method_mlp import MLPEnsemble
from src.preprocessing.scaler import prepare
from src.evaluation.metrics import ClassificationMetrics


# Limit CPU threads if needed for the current machine.
torch.set_num_threads(2)

# Load the existing MNIST files directly.
with np.load(path / "data/raw/mnist_train.npz", allow_pickle=False) as file:
    X_train = file["X"]
    y_train = file["y"]

with np.load(path / "data/raw/mnist_test.npz", allow_pickle=False) as file:
    X_test = file["X"]
    y_test = file["y"]

# Split the training set once and share the same preprocessing
# across all members. "raw" flattens images and scales pixels to [0, 1].
data = prepare(
    "raw",
    X_train,
    y_train,
    X_test,
    y_test,
    val_size=0.1,
)

ensemble = MLPEnsemble(
    sizes=(784, 128, 10),
    seeds=(42, 123, 2026),
    activation="relu",
    p_drop=0.0,
    init="he",
)

# Each member trains independently and restores its own best
# validation checkpoint through the existing MLP early stopping.
ensemble.fit(
    data["X_train"],
    data["y_train"],
    validation_data=(data["X_val"], data["y_val"]),
    epochs=30,
    batch_size=128,
    lr=0.05,
    momentum=0.9,
    weight_decay=1e-4,
    patience=5,
    verbose=True,
)

# Compare members on validation data while choosing configurations.
print("\nValidation results:")
for result in ensemble.evaluate_members(
    data["X_val"],
    data["y_val"],
):
    print(
        f"{result['model']:10s} | "
        f"loss={result['loss']:.4f} | "
        f"accuracy={result['accuracy']:.4%}"
    )

# Run this final evaluation only after configuration choices are fixed.
test_loss, test_accuracy = ensemble.evaluate(
    data["X_test"],
    data["y_test"],
)

test_predictions = ensemble.predict(data["X_test"]).numpy()

metrics = ClassificationMetrics(
    data["y_test"],
    test_predictions,
    labels=np.arange(10),
).evaluate()

print("\nFinal ensemble test results:")
print(f"Cross-entropy: {test_loss:.4f}")
print(f"Accuracy:      {test_accuracy:.4%}")
print(f"Macro F1:      {metrics['macro_f1']:.4f}")

# Training history for each member is retained separately.
first_member_history = ensemble.histories_[0]


Training member 1/3 (seed=42)
Epoch 01/30 | train loss 0.1612 acc 0.9533 | val loss 0.1868 acc 0.9463
Epoch 02/30 | train loss 0.1098 acc 0.9689 | val loss 0.1431 acc 0.9588
Epoch 03/30 | train loss 0.0805 acc 0.9769 | val loss 0.1185 acc 0.9657
Epoch 04/30 | train loss 0.0626 acc 0.9826 | val loss 0.1055 acc 0.9700
Epoch 05/30 | train loss 0.0548 acc 0.9846 | val loss 0.1015 acc 0.9725
Epoch 06/30 | train loss 0.0533 acc 0.9847 | val loss 0.1037 acc 0.9717
Epoch 07/30 | train loss 0.0471 acc 0.9869 | val loss 0.0988 acc 0.9718
Epoch 08/30 | train loss 0.0331 acc 0.9921 | val loss 0.0872 acc 0.9758
Epoch 09/30 | train loss 0.0318 acc 0.9921 | val loss 0.0891 acc 0.9760
Epoch 10/30 | train loss 0.0247 acc 0.9948 | val loss 0.0838 acc 0.9787
Epoch 11/30 | train loss 0.0224 acc 0.9956 | val loss 0.0827 acc 0.9775
Epoch 12/30 | train loss 0.0214 acc 0.9954 | val loss 0.0849 acc 0.9782
Epoch 13/30 | train loss 0.0187 acc 0.9966 | val loss 0.0873 acc 0.9770
Epoch 14/30 | train loss 0.0198 a